# 02 — Neo4j Graph Load · H&M In-Store Movement & Product Placement

**Session 2 Deliverable** — LOAD CSV + MERGE into Neo4j

| Step | What happens |
|------|--------------|
| 1 | Verify Neo4j connection |
| 2 | Create constraints + indexes |
| 3 | Load `Article` nodes via LOAD CSV + MERGE |
| 4 | Load `Customer` nodes via LOAD CSV + MERGE |
| 5 | Load `PURCHASED` relationships via LOAD CSV + MERGE |
| 6 | Verification Cypher queries + output |

**Pre-requisites:**
- Neo4j running on `bolt://localhost:7687` (Docker or Desktop)
- CSV files generated by `01_etl.ipynb` are in `data/neo4j/`
- `neo4j` Python driver installed: `pip install neo4j`

> **Docker quick-start** (if not already running):
> ```bash
> docker run -d \
>   --name neo4j \
>   -p 7474:7474 -p 7687:7687 \
>   -e NEO4J_AUTH=neo4j/password123 \
>   -v $(pwd)/data/neo4j:/var/lib/neo4j/import \
>   neo4j:5.18.0-community
> ```
> Then wait ~30 seconds for Neo4j to start before running this notebook.

In [4]:
# ── 0. Install / import ───────────────────────────────────────────────────────
!pip install neo4j pandas

from neo4j import GraphDatabase
import pandas as pd
from pathlib import Path
import time

# ── Connection config — update password if different ─────────────────────────
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "password123"   # ← change to match your docker/desktop password

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session() as s:
    result = s.run("RETURN 'Neo4j connected ✅' AS msg")
    print(result.single()['msg'])

# Helper: run a Cypher query and return a DataFrame
def cypher(query, params=None):
    with driver.session() as s:
        result = s.run(query, params or {})
        return pd.DataFrame([r.data() for r in result])

Neo4j connected ✅


In [5]:
# ── 1. Wipe existing data (clean slate for this session) ──────────────────────
# CAUTION: This deletes ALL nodes and relationships in the database.
# Only run this in your development/test Neo4j — not production.

print("Wiping existing graph data...")
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
print("  ✅  Graph cleared.")

Wiping existing graph data...
  ✅  Graph cleared.


In [6]:
# ── 2. Create constraints and indexes ────────────────────────────────────────
# Constraints ensure uniqueness AND auto-create a B-tree index (fast MERGE).
# MUST be created BEFORE loading — otherwise MERGE scans the entire graph.

constraints = [
    # Unique constraints (also create index automatically)
    "CREATE CONSTRAINT article_id IF NOT EXISTS "
    "FOR (a:Article)  REQUIRE a.articleId IS UNIQUE",

    "CREATE CONSTRAINT customer_id IF NOT EXISTS "
    "FOR (c:Customer) REQUIRE c.customerId IS UNIQUE",

    # Index on frequently queried properties
    "CREATE INDEX article_type IF NOT EXISTS "
    "FOR (a:Article)  ON (a.productType)",

    "CREATE INDEX article_section IF NOT EXISTS "
    "FOR (a:Article)  ON (a.storeSection)",

    "CREATE INDEX customer_age_band IF NOT EXISTS "
    "FOR (c:Customer) ON (c.ageBand)",

    "CREATE INDEX customer_member IF NOT EXISTS "
    "FOR (c:Customer) ON (c.memberStatus)",
]

with driver.session() as s:
    for stmt in constraints:
        s.run(stmt)
        label = stmt.split('\n')[0][:70]
        print(f"  ✅  {label}")

print("\nAll constraints and indexes created.")

  ✅  CREATE CONSTRAINT article_id IF NOT EXISTS FOR (a:Article)  REQUIRE a.
  ✅  CREATE CONSTRAINT customer_id IF NOT EXISTS FOR (c:Customer) REQUIRE c
  ✅  CREATE INDEX article_type IF NOT EXISTS FOR (a:Article)  ON (a.product
  ✅  CREATE INDEX article_section IF NOT EXISTS FOR (a:Article)  ON (a.stor
  ✅  CREATE INDEX customer_age_band IF NOT EXISTS FOR (c:Customer) ON (c.ag
  ✅  CREATE INDEX customer_member IF NOT EXISTS FOR (c:Customer) ON (c.memb

All constraints and indexes created.


In [11]:
# ── 3. Load Article nodes ─────────────────────────────────────────────────────
#
# LOAD CSV reads the file line by line.
# MERGE finds an existing node or creates a new one (idempotent).
# SET updates properties — safe to re-run.
# CALL { ... } IN TRANSACTIONS batches writes — critical for large files.
#
# Neo4j reads from its /import directory.
# The Docker run command above mounts data/neo4j → /var/lib/neo4j/import
#

ARTICLE_LOAD = """
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Article.csv' AS row
CALL {
    WITH row
    MERGE (a:Article { articleId: row.articleId })
    SET
        a.name          = row.name,
        a.productType   = row.productType,
        a.productGroup  = row.productGroup,
        a.colourGroup   = row.colourGroup,
        a.colour        = row.colour,
        a.department    = row.department,
        a.indexGroup    = row.indexGroup,
        a.section       = row.section,
        a.garmentGroup  = row.garmentGroup,
        a.storeSection  = row.storeSection,
        a.description   = row.description
} IN TRANSACTIONS OF 1000 ROWS
"""

print("Loading Article nodes...")
start = time.time()
with driver.session() as s:
    s.run(ARTICLE_LOAD)
elapsed = time.time() - start

count = cypher("MATCH (a:Article) RETURN COUNT(a) AS n").iloc[0,0]
print(f"  ✅  Article nodes loaded: {count:,}  ({elapsed:.1f}s)")

Loading Article nodes...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=3, column=1, offset=68>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 68, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nLOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Article.csv' AS row\nCALL {\n    WITH row\n    MERGE (a:Article { articleId: row.articleId })\n    SET\n        a.name          = row.name,\n        a.productType   = row.productType,\n        a.productGroup  = row.productGroup,\n        a.colourGroup   = row.colourGroup,\n        a.colour   

  ✅  Article nodes loaded: 105,542  (7.6s)


In [12]:
# ── 4. Load Customer nodes ────────────────────────────────────────────────────

CUSTOMER_LOAD = """
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Customer.csv' AS row
CALL {
    WITH row
    MERGE (c:Customer { customerId: row.customerId })
    SET
        c.age                = toInteger(row.age),
        c.ageBand            = row.ageBand,
        c.memberStatus       = row.memberStatus,
        c.newsFrequency      = row.newsFrequency,
        c.fnFlag             = toInteger(row.fnFlag),
        c.activeFlag         = toInteger(row.activeFlag),
        c.postalCode         = row.postalCode
} IN TRANSACTIONS OF 1000 ROWS
"""

print("Loading Customer nodes...")
start = time.time()
with driver.session() as s:
    s.run(CUSTOMER_LOAD)
elapsed = time.time() - start

count = cypher("MATCH (c:Customer) RETURN COUNT(c) AS n").iloc[0,0]
print(f"  ✅  Customer nodes loaded: {count:,}  ({elapsed:.1f}s)")

Loading Customer nodes...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=3, column=1, offset=69>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 69, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nLOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Customer.csv' AS row\nCALL {\n    WITH row\n    MERGE (c:Customer { customerId: row.customerId })\n    SET\n        c.age                = toInteger(row.age),\n        c.ageBand            = row.ageBand,\n        c.memberStatus       = row.memberStatus,\n        c.newsFrequency      = row.new

  ✅  Customer nodes loaded: 1,371,980  (45.4s)


In [13]:
# ── 5. Load PURCHASED relationships ──────────────────────────────────────────
#
# Each row in neo4j_rel_PURCHASED.csv becomes one relationship:
#   (:Customer { customerId }) -[:PURCHASED]-> (:Article { articleId })
#
# Properties on the relationship: txDate, yearMonth, price, channel
#
# NOTE: With 31M rows this takes several minutes.
# Batch size 5000 is a good balance between speed and memory.

PURCHASED_LOAD = """
LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_PURCHASED.csv' AS row
CALL {
    WITH row
    MATCH (c:Customer { customerId: row.customerId })
    MATCH (a:Article  { articleId:  row.articleId  })
    MERGE (c)-[r:PURCHASED]->(a)
    ON CREATE SET
        r.txDate    = date(row.txDate),
        r.yearMonth = row.yearMonth,
        r.price     = toFloat(row.price),
        r.channel   = row.channel
} IN TRANSACTIONS OF 5000 ROWS
"""

print("Loading PURCHASED relationships...")
print("(This may take several minutes for 31M rows — grab a coffee ☕)")
start = time.time()
with driver.session() as s:
    s.run(PURCHASED_LOAD)
elapsed = time.time() - start

count = cypher("MATCH ()-[r:PURCHASED]->() RETURN COUNT(r) AS n").iloc[0,0]
print(f"  ✅  PURCHASED relationships loaded: {count:,}  ({elapsed:.1f}s)")

Loading PURCHASED relationships...
(This may take several minutes for 31M rows — grab a coffee ☕)


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=3, column=1, offset=69>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 69, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nLOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_PURCHASED.csv' AS row\nCALL {\n    WITH row\n    MATCH (c:Customer { customerId: row.customerId })\n    MATCH (a:Article  { articleId:  row.articleId  })\n    MERGE (c)-[r:PURCHASED]->(a)\n    ON CREATE SET\n        r.txDate    = date(row.txDate),\n        r.yearMonth = row.yearMonth,\n       

  ✅  PURCHASED relationships loaded: 27,306,439  (2402.4s)


In [14]:
# ── 6. VERIFICATION — Graph summary ──────────────────────────────────────────
print("=" * 55)
print("  GRAPH SUMMARY")
print("=" * 55)

summary = cypher("""
    MATCH (n)
    WITH labels(n)[0] AS label, COUNT(n) AS nodeCount
    RETURN label, nodeCount
    ORDER BY nodeCount DESC
""")
print("\nNode counts:")
print(summary.to_string(index=False))

rel_summary = cypher("""
    MATCH ()-[r]->()
    RETURN TYPE(r) AS relType, COUNT(r) AS relCount
    ORDER BY relCount DESC
""")
print("\nRelationship counts:")
print(rel_summary.to_string(index=False))

  GRAPH SUMMARY

Node counts:
   label  nodeCount
Customer    1371980
 Article     105542

Relationship counts:
  relType  relCount
PURCHASED  27306439


In [15]:
# ── 7. VERIFICATION QUERY 1 — Sample Article nodes ────────────────────────────
print("=" * 55)
print("  VERIFICATION QUERY 1: Sample Article nodes")
print("=" * 55)

q1 = cypher("""
    MATCH (a:Article)
    RETURN
        a.articleId     AS articleId,
        a.name          AS name,
        a.productType   AS productType,
        a.storeSection  AS storeSection,
        a.colourGroup   AS colourGroup
    LIMIT 5
""")
print(q1.to_string(index=False))

  VERIFICATION QUERY 1: Sample Article nodes
 articleId              name productType storeSection colourGroup
0108775015         Strap top    Vest top   Womenswear       Black
0108775044         Strap top    Vest top   Womenswear       White
0108775051     Strap top (1)    Vest top   Womenswear   Off White
0110065001 OP T-shirt (Idro)         Bra   Womenswear       Black
0110065002 OP T-shirt (Idro)         Bra   Womenswear       White


In [16]:
# ── 8. VERIFICATION QUERY 2 — Sample Customer nodes ───────────────────────────
print("=" * 55)
print("  VERIFICATION QUERY 2: Sample Customer nodes")
print("=" * 55)

q2 = cypher("""
    MATCH (c:Customer)
    RETURN
        c.customerId    AS customerId,
        c.age           AS age,
        c.ageBand       AS ageBand,
        c.memberStatus  AS memberStatus,
        c.activeFlag    AS activeFlag
    LIMIT 5
""")
print(q2.to_string(index=False))

  VERIFICATION QUERY 2: Sample Customer nodes
                                                      customerId  age ageBand memberStatus  activeFlag
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657   49     40s       ACTIVE           0
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa   25     20s       ACTIVE           0
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318   24     20s       ACTIVE           0
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e   54     50s       ACTIVE           0
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a   52     50s       ACTIVE           1


In [17]:
# ── 9. VERIFICATION QUERY 3 — Traverse the graph ──────────────────────────────
# "Show me 5 customers and what they bought, with price and channel"

print("=" * 55)
print("  VERIFICATION QUERY 3: Customer → PURCHASED → Article")
print("=" * 55)

q3 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    RETURN
        c.customerId    AS customerId,
        c.ageBand       AS ageBand,
        a.name          AS articleName,
        a.productType   AS productType,
        r.txDate        AS txDate,
        r.price         AS price,
        r.channel       AS channel
    LIMIT 10
""")
print(q3.to_string(index=False))

  VERIFICATION QUERY 3: Customer → PURCHASED → Article
                                                      customerId ageBand articleName productType     txDate  price  channel
000f7535bdc611ad136a9f04746d6b1431f50a7f60fbbed14ea401d602badb69     40s   Strap top    Vest top 2019-07-14 0.0085 In_Store
001ae5408a043f64bccd32beffe2730151414cbdf18a6eb3cc8d30bdca605652     50s   Strap top    Vest top 2018-09-29 0.0068 In_Store
001ba9e81e13ce12a2585d9ebde923fe74429e9e12ea59f4f28a96267597fe52     20s   Strap top    Vest top 2019-05-23 0.0082   Online
0022a721371d5949d174ecba60346d89a9d6c08c0fba4f47b3b1e66b3fb58fd8     40s   Strap top    Vest top 2019-05-12 0.0085   Online
002323971cbd38fad4512d5114676e5e17eb262db02320419b45cc86ec09af87     20s   Strap top    Vest top 2018-12-06 0.0085   Online
0026b7d3b76ea96fcaebfc970961107f5c07dc91669144dd75f5e5d695b5fdfc     20s   Strap top    Vest top 2019-03-22 0.0085   Online
002da1d307ae48e1b8528792f62117e43a911ed075019547d64c091ee9e23e73     20s   St

In [18]:
# ── 10. VERIFICATION QUERY 4 — Top 10 most purchased articles ─────────────────
print("=" * 55)
print("  VERIFICATION QUERY 4: Top 10 most purchased articles")
print("=" * 55)

q4 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    RETURN
        a.articleId    AS articleId,
        a.name         AS name,
        a.productType  AS productType,
        a.storeSection AS storeSection,
        COUNT(r)       AS purchaseCount,
        ROUND(AVG(r.price), 4) AS avgPrice
    ORDER BY purchaseCount DESC
    LIMIT 10
""")
print(q4.to_string(index=False))

  VERIFICATION QUERY 4: Top 10 most purchased articles
 articleId                     name      productType  storeSection  purchaseCount  avgPrice
0706016001 Jade HW Skinny Denim TRS         Trousers Young Fashion          32251    0.0324
0372860001       7p Basic Shaftless            Socks    Womenswear          25559    0.0129
0706016002 Jade HW Skinny Denim TRS         Trousers Young Fashion          25485    0.0324
0610776002                Tilly (1)          T-shirt    Womenswear          22571    0.0081
0759871002               Tilda tank         Vest top Young Fashion          21613    0.0056
0372860002       7p Basic Shaftless            Socks    Womenswear          20038    0.0122
0464297007 Greta Thong Mynta Low 3p Underwear bottom    Womenswear          18554    0.0162
0720125001        SUPREME RW tights  Leggings/Tights         Sport          17611    0.0324
0673396002            Ringo hipbelt             Belt    Womenswear          17147    0.0130
0610776001               

In [19]:
# ── 11. VERIFICATION QUERY 5 — Customer purchase behaviour by age band ─────────
print("=" * 55)
print("  VERIFICATION QUERY 5: Purchases by age band + channel")
print("=" * 55)

q5 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    RETURN
        c.ageBand        AS ageBand,
        r.channel        AS channel,
        COUNT(r)         AS totalPurchases,
        COUNT(DISTINCT c.customerId) AS uniqueCustomers,
        ROUND(AVG(r.price), 4)       AS avgPrice
    ORDER BY ageBand, channel
""")
print(q5.to_string(index=False))

  VERIFICATION QUERY 5: Purchases by age band + channel
 ageBand  channel  totalPurchases  uniqueCustomers  avgPrice
     20s In_Store         3604099           310452    0.0229
     20s   Online         7803493           453486    0.0287
     30s In_Store         1451714           120724    0.0226
     30s   Online         4015929           192931    0.0297
     40s In_Store         1477114           109110    0.0229
     40s   Online         2720515           159137    0.0298
     50s In_Store         1573359           119923    0.0242
     50s   Online         2827239           177927    0.0319
 60_plus In_Store          456287            43516    0.0246
 60_plus   Online          765495            66983    0.0335
Under_20 In_Store          254127            32204    0.0221
Under_20   Online          357068            57084    0.0243


In [20]:
# ── 12. VERIFICATION QUERY 6 — Top store sections by revenue ──────────────────
print("=" * 55)
print("  VERIFICATION QUERY 6: Revenue by store section")
print("=" * 55)

q6 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    RETURN
        a.storeSection   AS storeSection,
        COUNT(r)         AS totalPurchases,
        COUNT(DISTINCT a.articleId) AS uniqueArticles,
        ROUND(SUM(r.price), 2)      AS totalRevenue,
        ROUND(AVG(r.price), 4)      AS avgPrice
    ORDER BY totalRevenue DESC
""")
print(q6.to_string(index=False))

  VERIFICATION QUERY 6: Revenue by store section
 storeSection  totalPurchases  uniqueArticles  totalRevenue  avgPrice
   Womenswear        17501935           39356     500289.25    0.0286
Young Fashion         6086327           15050     158498.18    0.0260
     Menswear         1533010           12457      42123.40    0.0275
        Sport         1100691            3381      31428.49    0.0286
     Kidswear         1084476           34303      20867.94    0.0192


In [21]:
# ── 13. Final summary + next steps ───────────────────────────────────────────
print("=" * 60)
print("  GRAPH LOAD COMPLETE — Session 2 Deliverable")
print("=" * 60)

final = cypher("""
    MATCH (n)
    WITH labels(n)[0] AS label, COUNT(n) AS cnt
    RETURN label, cnt
    UNION ALL
    MATCH ()-[r]->()
    RETURN TYPE(r) AS label, COUNT(r) AS cnt
""")
for _, row in final.iterrows():
    print(f"  {row['label']:<25}  {row['cnt']:>12,}")

print()
print("  Ready for Session 3:")
print("  - Add ProductGroup, Department, StoreSection nodes")
print("  - Add IN_GROUP, BELONGS_TO_DEPT relationships")
print("  - Add CO_PURCHASED edges from basket co-occurrence")
print("  - Run Neo4j GDS: PageRank, Louvain, Betweenness")

driver.close()

  GRAPH LOAD COMPLETE — Session 2 Deliverable
  Article                         105,542
  Customer                      1,371,980
  PURCHASED                    27,306,439

  Ready for Session 3:
  - Add ProductGroup, Department, StoreSection nodes
  - Add IN_GROUP, BELONGS_TO_DEPT relationships
  - Add CO_PURCHASED edges from basket co-occurrence
  - Run Neo4j GDS: PageRank, Louvain, Betweenness
